# Native Pythia optimizer-state reconstruction

**Grand Challenge release notebook.** This notebook reconstructs released GPT-NeoX Adam moments, validates the native packet contract, and measures CPS on historical optimizer state.


## Release contract

- **Scientific question:** Can the released native checkpoint be reconstructed and aligned safely enough for a historical CPS probe?
- **Default execution path:** Download the final Pythia-70M native packet, reconcile its metadata and moment coverage, then run the selected matrix-weight probe.
- **Evidence boundary:** The reader fails closed for unavailable parameter classes, ambiguous names, incompatible shapes, or inconsistent capacities.
- **Primary outputs:** Native download manifest, reconstruction contract, CPS evidence packet, diagnostic figures, and export archive.


## Interpretation checklist

- Confirm the resolved Hub revision and inferred training step before reconstruction.
- Read native moment coverage, name alignment, and shape-source metadata before interpreting the operator.
- A partial native packet supports only parameter classes whose moments are actually present.


## Operational warning

Native checkpoints are storage-heavy and checkpoint availability differs by model size. The notebook downloads only configuration, model-state metadata, and optimizer-state partitions selected by the repository helper. Confirm Colab disk capacity before continuing.

The **semantic training revision** and the **Hub revision that stores the native files** are not always the same. CPS records both and never silently substitutes a weight-only checkpoint for missing optimizer moments.


In [ ]:
import os, pathlib, subprocess, sys, time
from IPython.display import Markdown, display

REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")

print("[BOOT] Preparing the CPS repository", flush=True)
print(f"[BOOT] source={REPO_URL}", flush=True)
print(f"[BOOT] ref={GIT_REF}", flush=True)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", GIT_REF], check=True)
os.chdir(repo)
print("[BOOT] Installing CPS with Pythia and notebook dependencies", flush=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-e", ".[pythia,notebooks]"
], check=True)
print(f"[BOOT] Ready: {repo}", flush=True)

# Editable installs write a .pth file, but the running Colab kernel does not
# automatically reprocess newly-created .pth files. Put the source tree on
# sys.path explicitly so the very next cell can import CPS without a restart.
import importlib
src_dir = repo / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
importlib.invalidate_caches()
import cps
print(f"[BOOT] CPS import verified from {cps.__file__}", flush=True)

In [ ]:
from cps.notebook import stage_banner
stage_banner(
    "0",
    "Record the runtime contract",
    objective="Expose the software and accelerator environment before scientific execution.",
    deliverable="A visible runtime inventory for reproducibility.",
)
from cps.notebook import show_environment
runtime = show_environment()

## Stage 1 — resolve, download, and inventory the native checkpoint

`CPS_NATIVE_REVISION` names the **training checkpoint** being studied. The default is `step143000`.

The Transformers-compatible Pythia weight repository exposes historical `stepN` branches. The current native GPT-NeoX optimizer-state repository may expose its final checkpoint only as `main`. The downloader queries the repository refs first; when `step143000` is absent but `main` is present, it records an explicit final-step resolution:

```text
requested revision: step143000
resolved native revision: main
training step: 143000
```

Other missing revisions fail closed. CPS does not fabricate optimizer moments from model weights.


In [ ]:
from cps.notebook import stage_banner
stage_banner('1', 'resolve, download, and inventory the native checkpoint', objective='CPSNATIVEREVISION names the training checkpoint being studied. The default is step143000.', deliverable="The artifacts and console evidence described in this stage.")

import os, pathlib
from cps.pythia.checkpoints import download_native_checkpoint

requested_revision = os.environ.get("CPS_NATIVE_REVISION", "step143000")
target = pathlib.Path(f"/content/native-pythia-70m-{requested_revision}")
print(
    f"[NATIVE] requesting training revision={requested_revision} "
    f"from EleutherAI/neox-ckpt-pythia-70m",
    flush=True,
)
result = download_native_checkpoint(
    "EleutherAI/neox-ckpt-pythia-70m",
    requested_revision,
    target,
)
print(f"[NATIVE] requested revision={result.requested_revision}", flush=True)
print(f"[NATIVE] resolved Hub revision={result.resolved_revision}", flush=True)
print(f"[NATIVE] inferred training step={result.training_step}", flush=True)
if result.resolution_note:
    print(f"[NATIVE] resolution: {result.resolution_note}", flush=True)
print(
    "[NATIVE] available branches="
    + (", ".join(result.available_branches) or "<ref discovery unavailable>"),
    flush=True,
)

total_bytes = sum(pathlib.Path(path).stat().st_size for path in result.files)
print(f"[NATIVE] files={len(result.files)}; bytes={total_bytes:,}", flush=True)
for path in result.files:
    p = pathlib.Path(path)
    print(f"  {p.relative_to(target)}  {p.stat().st_size / 2**20:.2f} MiB", flush=True)

## Stage 2 — bind and validate the native moments

The optimizer step number is derived from the semantic checkpoint revision. Historical GPT-NeoX packets vary in metadata, optimizer-state layout, and parameter naming. The raw pipeline checkpoint can name the first Transformer block with a pipeline-stage path such as `module.sequential.2.attention.dense.weight`, while Hugging Face names the same tensor `gpt_neox.layers.0.attention.dense.weight`. CPS reconciles moment capacities first, then aligns requested names by exact prefix variants or by the shape-checked occurrence of a semantic suffix across Transformer layers. Every alias and any required two-dimensional transpose is printed and recorded. Ambiguous mappings fail closed.


In [ ]:
from cps.notebook import stage_banner
stage_banner('2', 'bind and validate the native moments', objective='The optimizer step number is derived from the semantic checkpoint revision. Historical GPTNeoX packets vary in metadata, optimizerstate layout, and parameter naming. The raw pipeline checkpoint can name the first Transformer block with a pipelinestage path such as module.sequential.2.attention.dense.weight, while Hugging Face names the same tensor gptneox.layers.0.attention.dense.weight. CPS reconciles moment capacities first, then aligns requested names by exact prefix variants or by the shapechecked occurrence of a semantic suffix across Transformer layers. Every alias and any required twodimensional transpose is printed and recorded. Ambiguous mappings fail closed.', deliverable="The artifacts and console evidence described in this stage.")

import dataclasses
from cps.notebook import show_config
from cps.pythia.config import load_probe_config

base = load_probe_config("subjects/pythia/configs/pythia_70m_native.yaml")
if result.training_step is None:
    raise ValueError(
        "The native checkpoint training step could not be inferred. "
        "Set CPS_NATIVE_REVISION to a stepN revision or add an explicit mapping."
    )

step_number = max(1, result.training_step)
state = dataclasses.replace(
    base.state,
    native_checkpoint_dir=str(target),
    step=step_number,
)
model = dataclasses.replace(base.model, revision=requested_revision)
config = dataclasses.replace(base, state=state, model=model)

print(
    f"[NATIVE] model weights revision={config.model.revision}; "
    f"native files revision={result.resolved_revision}; optimizer step={step_number}",
    flush=True,
)
show_config(config)

In [ ]:
import importlib
import cps.pythia.native_state_packet as native_packet
import cps.pythia.runner as pythia_runner

# Colab keeps imported modules alive across git pulls. Reload the packet
# reader and bind it explicitly so a stale runner cannot silently select
# the older complete-group reconstruction path.
native_packet = importlib.reload(native_packet)
pythia_runner.reconstruct_zero_adam_state = native_packet.reconstruct_zero_adam_state
reader = pythia_runner.reconstruct_zero_adam_state
print(f"[NATIVE] active reader={reader.__module__}.{reader.__name__}", flush=True)
if reader is not native_packet.reconstruct_zero_adam_state:
    raise RuntimeError("packet-aware native reader was not bound into the runner")

print("[NATIVE] Reconstructing Adam moments from ZeRO partitions.", flush=True)
print("[NATIVE] The reader will classify complete, decay-only, or no-decay-only coverage", flush=True)
print("[NATIVE] and will fail closed for selected parameters outside the visible packet.", flush=True)
output = pythia_runner.run_probe(config)
print(f"[NATIVE] evidence root={output}", flush=True)


In [ ]:
from cps.notebook import display_probe_summary
summary = display_probe_summary(output)

## Final stage — export the evidence packet

Every release notebook ends with the same preservation step. The archive contains the evidence produced in this runtime and is suitable for Colab CLI retrieval or manual download.


In [ ]:
from cps.notebook import export_artifacts, stage_banner

stage_banner(
    "EXPORT",
    "Package the evidence",
    objective="Collect the run artifacts into one portable archive.",
    deliverable="/content/cps-export.zip",
)
archive = export_artifacts()
print(f"[EXPORT] archive={archive}", flush=True)
